# Regressão linear 

In [38]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.tools.sm_exceptions import PerfectSeparationError


In [39]:
Leitura do dataset.

SyntaxError: invalid syntax (1594417268.py, line 1)

In [ ]:
caminho = r'C:\Users\saulosgil\OneDrive\Documentos\health_outcomes_workers\data\df_analise.csv'
df = pd.read_csv(caminho)

df.head()

,idade,sexo,estado_civil,escolaridade,renda_mensal,cor_raca,tempo_trab_anos,funcao_trab,carga_horaria_horas_sem,outro_trab,...,Diuretics,Beta_blockers,ACE_inhibitors,Metformin,CCB,whoqol_fisico,whoqol_psicologico,whoqol_social,whoqol_ambiente,whoqol_n_validos
0,34,feminino,separado,médio completo,1 a 3 salários mínimos,preto (a),1.0,cooperado,40,sim,...,0,0,0,0,0,71.43,70.83,83.33,78.12,26
1,59,masculino,divorciado,médio completo,mais de 3 salários,pardo (a),2.0,cooperado,56,não,...,0,0,0,0,0,100.00,83.33,100.00,93.75,26
2,27,masculino,solteiro,superior completo,mais de 3 salários,pardo (a),1.0,cooperado,44,não,...,0,0,0,0,0,78.57,58.33,75.00,75.00,26
3,40,masculino,solteiro,médio completo,1 a 3 salários mínimos,preto (a),1.0,cooperado,45,não,...,0,0,0,0,0,78.57,95.83,100.00,78.12,26
4,23,masculino,solteiro,médio completo,mais de 3 salários,preto (a),2.0,cooperado,45,não,...,0,0,0,0,0,75.00,83.33,66.67,90.62,26


Ajuste das variáveis para aplicar no modelo.

In [ ]:
df['funcao_trab'] = df['funcao_trab'].replace({
    'cooperado':   'Cooperative waste pickers',
    'individual':  'Independent waste pickers',
    'Cooperative': 'Cooperative waste pickers',
    'Independent': 'Independent waste pickers',
})

# Binária: < 65 = 1, >= 65 = 0, ausente = NaN
df['idade_cat'] = np.where(df['idade'] < 65, '< 65 anos', '>= 65 anos')

# Regressão logistica  

1. Desfechos
    - `obesidade`;                 
    - `aumentado_cmr`.

In [42]:
ref   = "C(funcao_trab, Treatment('Cooperative waste pickers'))"
termo = f"{ref}[T.Independent waste pickers]"

# Desfechos BINÁRIOS (0/1)
desfechos_bin = {
    'obesidade':     'Obesity',
    'aumentado_cmr': 'Increased cardiometabolic risk',
}

# Conjunto mínimo de ajuste (confundidores plausíveis)

resultados_logit = []

for var, rotulo in desfechos_bin.items():
    formula = f"{var} ~ {ref}"

    try:
        m = smf.logit(formula, data=df).fit(disp=False, maxiter=100)
    except (PerfectSeparationError, np.linalg.LinAlgError) as e:
        print(f'⚠ {var}: modelo não estimável ({e})')
        continue

    if not m.mle_retvals['converged']:
        print(f'⚠ {var}: o modelo não convergiu, interprete com cautela')

    # Eventos por parâmetro (regra prática: ≥ 10)
    eventos = int(min(m.model.endog.sum(), m.nobs - m.model.endog.sum()))
    epv = eventos / (len(m.params) - 1)
    if epv < 10:
        print(f'⚠ {var}: {epv:.1f} eventos por variável (< 10)')

    # Odds ratio e IC 95%
    or_val     = np.exp(m.params[termo])
    lo, hi     = np.exp(m.conf_int().loc[termo])
    p          = m.pvalues[termo]

    resultados_logit.append({
        'Outcome': rotulo,
        'OR (IC 95%)': f'{or_val:.2f} ({lo:.2f} a {hi:.2f})',
        'P': round(p, 4),
        'n': int(m.nobs),
        'Eventos': eventos,
        'Pseudo-R² (McFadden)': round(m.prsquared, 2),
    })

tabela_logit = pd.DataFrame(resultados_logit)
tabela_logit

,Outcome,OR (IC 95%),P,n,Eventos,Pseudo-R² (McFadden)
0,Obesity,0.33 (0.09 a 1.21),0.0951,44,20,0.05
1,Increased cardiometabolic risk,0.32 (0.08 a 1.28),0.1074,44,12,0.05


# Regressão linear ajustada 

1. Desfechos
    - `obesidade`;                 
    - `aumentado_cmr`.

2. Covariáveis
    - `sexo`;                 
    - `idade_cat`;                  
    - `cor_raca`;        
    - `renda_mensal`; 
    - `DM2`;        
    - `HAS`.

In [41]:
ref   = "C(funcao_trab, Treatment('Cooperative waste pickers'))"
termo = f"{ref}[T.Independent waste pickers]"

# Desfechos BINÁRIOS (0/1)
desfechos_bin = {
    'obesidade':     'Obesity',
    'aumentado_cmr': 'Increased cardiometabolic risk',
}

# Conjunto mínimo de ajuste (confundidores plausíveis)
covariaveis = ['C(sexo)', 'C(idade_cat)']

resultados_logit = []

for var, rotulo in desfechos_bin.items():
    formula = f"{var} ~ {ref} + {' + '.join(covariaveis)}"

    try:
        m = smf.logit(formula, data=df).fit(disp=False, maxiter=100)
    except (PerfectSeparationError, np.linalg.LinAlgError) as e:
        print(f'⚠ {var}: modelo não estimável ({e})')
        continue

    if not m.mle_retvals['converged']:
        print(f'⚠ {var}: o modelo não convergiu, interprete com cautela')

    # Eventos por parâmetro (regra prática: ≥ 10)
    eventos = int(min(m.model.endog.sum(), m.nobs - m.model.endog.sum()))
    epv = eventos / (len(m.params) - 1)
    if epv < 10:
        print(f'⚠ {var}: {epv:.1f} eventos por variável (< 10)')

    # Odds ratio e IC 95%
    or_val     = np.exp(m.params[termo])
    lo, hi     = np.exp(m.conf_int().loc[termo])
    p          = m.pvalues[termo]

    resultados_logit.append({
        'Outcome': rotulo,
        'OR (IC 95%)': f'{or_val:.2f} ({lo:.2f} a {hi:.2f})',
        'P': round(p, 4),
        'n': int(m.nobs),
        'Eventos': eventos,
        'Pseudo-R² (McFadden)': round(m.prsquared, 2),
    })

tabela_logit = pd.DataFrame(resultados_logit)
tabela_logit

⚠ obesidade: 6.7 eventos por variável (< 10)
⚠ aumentado_cmr: 4.0 eventos por variável (< 10)


,Outcome,OR (IC 95%),P,n,Eventos,Pseudo-R² (McFadden)
0,Obesity,0.23 (0.05 a 1.00),0.0493,44,20,0.15
1,Increased cardiometabolic risk,0.24 (0.05 a 1.09),0.0648,44,12,0.13
